Karen Torres Feijoo
Práctica- Redes Neuronales Artificiales

In [39]:
import numpy as np
#Aplicamos la regla de actualización de pesos- COMPUERTA AND
def escalon(z):
    return 1 if z >= 0 else 0

In [40]:
class Perceptron:
    def __init__(self, n_entradas, lr=0.1):  # 4 espacios
        self.w = np.zeros(n_entradas)         # 8 espacios
        self.b = 0
        self.lr = lr

    def forward(self, x):                     # 4 espacios
        z = np.dot(self.w, x) + self.b       # 8 espacios
        return escalon(z)

    def backward(self, x, y_real):            # 4 espacios
        y_pred = self.forward(x)              # 8 espacios
        error = y_real - y_pred
        self.w += self.lr * error * x
        self.b += self.lr * error
        return error

In [41]:
X = np.array([[0,0], [0,1], [1,0], [1,1]])
y = np.array([0, 0, 0, 1])

p = Perceptron(n_entradas=2, lr=0.1)

for epoca in range(100):
    errores = 0
    for i in range(len(X)):
        error = p.backward(X[i], y[i])
        if error != 0:
            errores += 1

    print("Época:", epoca, "| Pesos:", p.w, "| Bias:", p.b)

    if errores == 0:
        print("¡Convergió en época!", epoca)
        break

Época: 0 | Pesos: [0.1 0.1] | Bias: 0.0
Época: 1 | Pesos: [0.2 0.1] | Bias: -0.1
Época: 2 | Pesos: [0.2 0.1] | Bias: -0.20000000000000004
Época: 3 | Pesos: [0.2 0.1] | Bias: -0.20000000000000004
¡Convergió en época! 3


In [42]:
x1 = np.array([0, 1])
x2 = (-p.b - p.w[0] * x1) / p.w[1]
print("Frontera de decisión:")
print("Cuando x1=0, x2=", x2[0])
print("Cuando x1=1, x2=", x2[1])

Frontera de decisión:
Cuando x1=0, x2= 2.0000000000000004
Cuando x1=1, x2= 2.7755575615628914e-16


In [48]:
def relu(z):
  return np.maximum(0,z)

def relu_derivada(z):
  return (z > 0).astype(int)


def softmax(z):
  e = np.exp(z)
  return e / np.sum(e, axis = 1, keepdims= True)


In [49]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import MinMaxScaler


iris = load_iris()
X = iris.data        # 4 características
y = iris.target      # 3 clases (0, 1, 2)

# Normalizar entre 0 y 1
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)
print("Clases:", np.unique(y))

Forma de X: (150, 4)
Forma de y: (150,)
Clases: [0 1 2]


In [50]:

from sklearn.preprocessing import MinMaxScaler
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data        # 4 características
y = iris.target      # 3 clases (0, 1, 2)

# Normalizar entre 0 y 1
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)
print("Clases:", np.unique(y))

Forma de X: (150, 4)
Forma de y: (150,)
Clases: [0 1 2]


In [51]:
def one_hot(y, n_clases=3):
    m = len(y)
    Y = np.zeros((m, n_clases))
    Y[np.arange(m), y] = 1
    return Y

Y = one_hot(y)
print("Forma de Y:", Y.shape)
print("Primeros 5 ejemplos:")
print(Y[:5])

Forma de Y: (150, 3)
Primeros 5 ejemplos:
[[1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]]


## Parte 2: Red Neuronal MLP - Dataset Iris
- Arquitectura: [4, 8, 3] → 4 entradas, 8 neuronas ocultas, 3 salidas
- Capa oculta: ReLU (evita el problema del gradiente desvaneciente)
- Capa salida: Softmax (convierte salidas en probabilidades)
- Learning rate: 0.1 (balance entre velocidad y estabilidad)
- Épocas: 1000 (la red estabiliza alrededor de época 600)

In [52]:
class MLP:
    def __init__(self, n_capas, lr=0.1):
        self.lr = lr
        self.W = []
        self.b = []
        for i in range(len(n_capas) - 1):
            w = np.random.randn(n_capas[i], n_capas[i+1]) * 0.1
            b = np.zeros((1, n_capas[i+1]))
            self.W.append(w)
            self.b.append(b)

    def forward(self, X):
        self.A = [X]
        self.Z = []
        z1 = np.dot(X, self.W[0]) + self.b[0]
        a1 = relu(z1)
        self.Z.append(z1)
        self.A.append(a1)
        z2 = np.dot(a1, self.W[1]) + self.b[1]
        a2 = softmax(z2)
        self.Z.append(z2)
        self.A.append(a2)
        return a2

    def backward(self, Y):
        m = Y.shape[0]
        delta2 = self.A[-1] - Y
        dW2 = np.dot(self.A[-2].T, delta2) / m
        db2 = np.sum(delta2, axis=0, keepdims=True) / m
        delta1 = np.dot(delta2, self.W[1].T) * relu_derivada(self.Z[0])
        dW1 = np.dot(self.A[0].T, delta1) / m
        db1 = np.sum(delta1, axis=0, keepdims=True) / m
        self.W[1] -= self.lr * dW2
        self.b[1] -= self.lr * db2
        self.W[0] -= self.lr * dW1
        self.b[0] -= self.lr * db1

    def predecir(self, X):
        return np.argmax(self.forward(X), axis=1)

In [53]:
np.random.seed(42)
red = MLP(n_capas=[4, 8, 3], lr=0.1)

for epoca in range(1000):
    salida = red.forward(X)
    red.backward(Y)

    if epoca % 200 == 0:
        predicciones = red.predecir(X)
        accuracy = np.mean(predicciones == y) * 100
        print(f"Época {epoca} | Accuracy: {accuracy:.1f}%")

Época 0 | Accuracy: 33.3%
Época 200 | Accuracy: 76.0%
Época 400 | Accuracy: 93.3%
Época 600 | Accuracy: 96.0%
Época 800 | Accuracy: 96.0%
